In [44]:
from pathlib import Path

import numpy as np
import pandas as pd

from arch import arch_model

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FINAL_DIR = PROJECT_ROOT / "data" / "final"

df = pd.read_csv(
    FINAL_DIR / "daily_volatility_engineered_provisional.csv",
    parse_dates=["Date"]
)

df = df.sort_values("Date").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

Dataset shape: (4024, 15)
Date range: 2010-01-04 00:00:00 to 2025-12-31 00:00:00


In [45]:
train_returns = (
    df.loc[
        df["Date"] < "2020-01-01",
        "Log_Return"
    ]
    .dropna()
    .to_numpy() * 100
)

print("Training observations:", len(train_returns))

print("Mean return (%):", train_returns.mean())
print("Standard deviation (%):", train_returns.std())

Training observations: 2515
Mean return (%): 0.041664550932619955
Standard deviation (%): 0.9315534081586319


In [46]:
garch = arch_model(
    train_returns,
    mean="Constant",
    vol="GARCH",
    p=1,
    q=1,
    dist="normal",
    rescale=False
)

garch_fit = garch.fit(
    disp="off"
)

print(garch_fit.summary())

                     Constant Mean - GARCH Model Results                      
Dep. Variable:                      y   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -3005.08
Distribution:                  Normal   AIC:                           6018.16
Method:            Maximum Likelihood   BIC:                           6041.48
                                        No. Observations:                 2515
Date:                Fri, Oct 09 2026   Df Residuals:                     2514
Time:                        11:06:24   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0789  1.399e-02      5.644  1.665e-08 [5.152e-0

In [47]:
HORIZON = 20

forecast = garch_fit.forecast(
    horizon=HORIZON,
    reindex=False
)

daily_variances = forecast.variance.iloc[-1].to_numpy()

# GARCH was fitted on returns multiplied by 100.
# Divide aggregated variance by 100^2 before square rooting.
forecast_rv_20d = (
    np.sqrt(daily_variances.sum()) / 100
)

print("=" * 60)
print("GARCH(1,1) — INITIAL 20-DAY FORECAST")
print("=" * 60)

print(f"Forecast 20-day realised volatility: {forecast_rv_20d:.6f}")

GARCH(1,1) — INITIAL 20-DAY FORECAST
Forecast 20-day realised volatility: 0.031154


In [48]:
# =========================================================
# STEP 5: PREPARE GARCH VALIDATION DATA
# =========================================================

returns_df = df[
    ["Date", "Log_Return", "RV_Future_20D"]
].copy()

returns_df = returns_df.dropna(
    subset=["Log_Return"]
).reset_index(drop=True)

returns_df["Return_Pct"] = (
    returns_df["Log_Return"] * 100
)

validation_mask = (
    (returns_df["Date"] >= "2020-01-01") &
    (returns_df["Date"] < "2023-01-01")
)

validation_garch = returns_df.loc[
    validation_mask
].copy()

print("Validation observations:", len(validation_garch))

print("First validation date:", validation_garch["Date"].min())
print("Last validation date:", validation_garch["Date"].max())

assert len(validation_garch) == 756

print("GARCH validation data prepared!")

Validation observations: 756
First validation date: 2020-01-02 00:00:00
Last validation date: 2022-12-30 00:00:00
GARCH validation data prepared!


In [49]:
# =========================================================
# STEP 6: EXTRACT GARCH PARAMETERS
# =========================================================

mu = garch_fit.params["mu"]
omega = garch_fit.params["omega"]
alpha = garch_fit.params["alpha[1]"]
beta = garch_fit.params["beta[1]"]

print("Mu:", mu)
print("Omega:", omega)
print("Alpha:", alpha)
print("Beta:", beta)

print("Persistence:", alpha + beta)

assert omega > 0
assert alpha >= 0
assert beta >= 0
assert alpha + beta < 1

print("\nGARCH parameter checks passed!")

Mu: 0.0789371481876778
Omega: 0.036845489812747614
Alpha: 0.17116065930946198
Beta: 0.7898304060852295
Persistence: 0.9609910653946915

GARCH parameter checks passed!


In [50]:
# =========================================================
# STEP 7: CORRECTED ROLLING-ORIGIN GARCH FORECASTS
# =========================================================

HORIZON = 20

all_returns = returns_df["Return_Pct"].to_numpy()

# Fitted conditional variances for the training period
training_variances = (
    np.asarray(garch_fit.conditional_volatility) ** 2
)

# Locate the last training observation
last_training_position = returns_df.index[
    returns_df["Date"] < "2020-01-01"
][-1]

# Variance of the final training observation
current_variance = training_variances[-1]

# Long-run conditional variance
long_run_variance = omega / (1 - alpha - beta)

garch_forecasts = []
garch_dates = []

# Begin with the first observation after training
for i in range(
    last_training_position + 1,
    len(returns_df)
):

    # Update conditional variance using return from day i-1.
    # This gives the conditional variance for day i.
    previous_return = all_returns[i - 1]

    current_variance = (
        omega
        + alpha * (previous_return - mu) ** 2
        + beta * current_variance
    )

    # At the close of day i, the return for day i is known.
    # Calculate conditional variance for day i+1.
    next_variance = (
        omega
        + alpha * (all_returns[i] - mu) ** 2
        + beta * current_variance
    )

    # Forecast conditional variances for days i+1 ... i+20.
    forecast_variances = []

    variance_h = next_variance

    for h in range(1, HORIZON + 1):

        if h > 1:
            variance_h = (
                omega
                + (alpha + beta) * variance_h
            )

        forecast_variances.append(variance_h)

    forecast_rv = (
        np.sqrt(np.sum(forecast_variances)) / 100
    )

    garch_forecasts.append(forecast_rv)
    garch_dates.append(returns_df.loc[i, "Date"])

garch_forecast_df = pd.DataFrame({
    "Date": garch_dates,
    "GARCH_Predicted": garch_forecasts
})

garch_validation = validation_garch.merge(
    garch_forecast_df,
    on="Date",
    how="left"
)

assert len(garch_validation) == 756
assert garch_validation["GARCH_Predicted"].notna().all()

print("Corrected GARCH forecasts generated.")

Corrected GARCH forecasts generated.


In [51]:
# =========================================================
# STEP 8: EVALUATE GARCH VALIDATION PERFORMANCE
# =========================================================

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

def evaluate_forecast(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(y_true, y_pred)

    epsilon = 1e-4

    safe_mape = np.mean(
        np.abs(y_true - y_pred) /
        np.maximum(np.abs(y_true), epsilon)
    ) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "Safe_MAPE": safe_mape
    }


garch_metrics = evaluate_forecast(
    garch_validation["RV_Future_20D"],
    garch_validation["GARCH_Predicted"]
)

print("=" * 60)
print("GARCH ROLLING-ORIGIN VALIDATION RESULTS")
print("=" * 60)

for metric, value in garch_metrics.items():
    print(f"{metric}: {value:.6f}")

GARCH ROLLING-ORIGIN VALIDATION RESULTS
RMSE: 0.034771
MAE: 0.019197
Safe_MAPE: 29.495222


In [52]:
# =========================================================
# STEP 9: COMPARE ALL MODELS
# =========================================================

comparison = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "RMSE": 0.035895,
        "MAE": 0.018616,
        "Safe_MAPE": 27.730350
    },
    {
        "Model": "Random Forest",
        "RMSE": 0.038677,
        "MAE": 0.019927,
        "Safe_MAPE": 27.855195
    },
    {
        "Model": "XGBoost",
        "RMSE": 0.037992,
        "MAE": 0.019562,
        "Safe_MAPE": 27.495317
    },
    {
        "Model": "Persistence",
        "RMSE": 0.043013,
        "MAE": 0.023945,
        "Safe_MAPE": 37.751981
    },
    {
        "Model": "GARCH(1,1)",
        **garch_metrics
    }
])

comparison = comparison.sort_values(
    "RMSE"
).reset_index(drop=True)

display(comparison.round(6))

,Model,RMSE,MAE,Safe_MAPE
0,"GARCH(1,1)",0.034771,0.019197,29.495222
1,Linear Regression,0.035895,0.018616,27.730350
2,XGBoost,0.037992,0.019562,27.495317
3,Random Forest,0.038677,0.019927,27.855195
4,Persistence,0.043013,0.023945,37.751981


In [53]:
# =========================================================
# STEP 10: SAVE GARCH RESULTS
# =========================================================

TABLES_DIR = PROJECT_ROOT / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

garch_validation.to_csv(
    TABLES_DIR / "garch_validation_predictions.csv",
    index=False
)

comparison.to_csv(
    TABLES_DIR / "garch_model_comparison.csv",
    index=False
)

print("GARCH predictions saved!")
print("Model comparison saved!")

GARCH predictions saved!
Model comparison saved!


In [54]:
# =========================================================
# STEP 11: CHECK FIRST ROLLING-ORIGIN FORECAST
# =========================================================

first_validation_date = validation_garch["Date"].iloc[0]

first_validation_position = returns_df.index[
    returns_df["Date"] == first_validation_date
][0]

print("First validation date:", first_validation_date)

print(
    "First GARCH forecast:",
    garch_validation["GARCH_Predicted"].iloc[0]
)

print(
    "Minimum forecast:",
    garch_validation["GARCH_Predicted"].min()
)

print(
    "Maximum forecast:",
    garch_validation["GARCH_Predicted"].max()
)

assert (
    garch_validation["GARCH_Predicted"] > 0
).all()

assert np.isfinite(
    garch_validation["GARCH_Predicted"]
).all()

print("\nBasic GARCH forecast checks passed!")

First validation date: 2020-01-02 00:00:00
First GARCH forecast: 0.032765713684793
Minimum forecast: 0.03015430423958159
Maximum forecast: 0.2880276761523194

Basic GARCH forecast checks passed!
